# Flipkart Gridlock 2.0 — V6 Grandmaster Architecture
## Two-Tower Logarithmic Target Transformation & 70/30 Fusion

**V6 Key Upgrades:**
1. **Target Compression:** Applies `log1p` to the target array to mitigate outlier spike penalties during gradient updates.
2. **Exponential Reversion:** Applies `expm1` to safely map log-space predictions back to raw physical capacity constraints.
3. **Asymmetric Meta-Blend:** Weighs the Generalization Tower at 70% and Deep Memory at 30% for optimal leaderboard scaling.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import pygeohash as pgh
from scipy.optimize import minimize
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
np.random.seed(42)

print(" [SYSTEM] V6 Initialized. Target Transformation active.")

 [SYSTEM] V6 Initialized. Target Transformation active.


In [2]:
print(" [INGESTION] Loading datasets and generating Log-Space Targets...")
POSSIBLE_ROUTES = [".", "data/raw", "../../data/raw"]
data_path = next((r for r in POSSIBLE_ROUTES if os.path.exists(os.path.join(r, "train.csv"))), None)

raw_train = pd.read_csv(os.path.join(data_path, "train.csv"))
raw_test = pd.read_csv(os.path.join(data_path, "test.csv"))

# V6 Core Change: Create both Raw and Log-transformed Targets
y_raw = raw_train['demand'].values
y_log = np.log1p(y_raw) # Compressed target for training
test_submission_index = raw_test['Index'].values

X_train_base = raw_train.drop(columns=['demand'], errors='ignore')
X_test_base = raw_test.copy()

 [INGESTION] Loading datasets and generating Log-Space Targets...


In [3]:
print(" [ENGINEERING] Building multi-day memory lags (24H & 48H)...")

lag_24 = raw_train[['geohash', 'day', 'timestamp', 'demand']].copy()
lag_24['day'] = lag_24['day'] + 1
lag_24.rename(columns={'demand': 'lag_demand_24h'}, inplace=True)

lag_48 = raw_train[['geohash', 'day', 'timestamp', 'demand']].copy()
lag_48['day'] = lag_48['day'] + 2
lag_48.rename(columns={'demand': 'lag_demand_48h'}, inplace=True)

X_train_lag = X_train_base.merge(lag_24, on=['geohash', 'day', 'timestamp'], how='left')
X_train_lag = X_train_lag.merge(lag_48, on=['geohash', 'day', 'timestamp'], how='left')

X_test_lag = X_test_base.merge(lag_24, on=['geohash', 'day', 'timestamp'], how='left')
X_test_lag = X_test_lag.merge(lag_48, on=['geohash', 'day', 'timestamp'], how='left')

global_mean = y_raw.mean()
for col in ['lag_demand_24h', 'lag_demand_48h']:
    X_train_lag[col] = X_train_lag[col].fillna(global_mean)
    X_test_lag[col] = X_test_lag[col].fillna(global_mean)

 [ENGINEERING] Building multi-day memory lags (24H & 48H)...


In [4]:
print(" [ENGINEERING] Synthesizing cyclical time and spatial features...")

def extract_features(df):
    df_feat = df.copy()
    
    df_feat['Temperature'] = df_feat['Temperature'].fillna(df_feat['Temperature'].median())
    df_feat['Weather'] = df_feat['Weather'].fillna('Unknown')
    df_feat['RoadType'] = df_feat['RoadType'].fillna('Unknown')
    
    time_split = df_feat['timestamp'].str.split(':', expand=True).astype(int)
    df_feat['ts_minutes'] = time_split[0] * 60 + time_split[1]
    df_feat['hour'] = time_split[0]
    df_feat['time_slot_15m'] = df_feat['ts_minutes'] // 15
    
    df_feat['hour_sin'] = np.sin(2 * np.pi * df_feat['hour'] / 24.0)
    df_feat['hour_cos'] = np.cos(2 * np.pi * df_feat['hour'] / 24.0)
    df_feat['min_sin'] = np.sin(2 * np.pi * df_feat['ts_minutes'] / 1440.0)
    df_feat['min_cos'] = np.cos(2 * np.pi * df_feat['ts_minutes'] / 1440.0)
    
    df_feat['is_peak_am'] = df_feat['hour'].between(7, 9).astype(int)
    df_feat['is_peak_pm'] = df_feat['hour'].between(17, 19).astype(int)
    df_feat['is_weekend'] = (df_feat['day'] % 7).isin([5, 6]).astype(int)
    df_feat['is_rush_hour'] = df_feat['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
    
    coords = df_feat['geohash'].apply(lambda x: pgh.decode(x) if isinstance(x, str) else (np.nan, np.nan))
    df_feat['lat'] = coords.apply(lambda x: x[0])
    df_feat['lon'] = coords.apply(lambda x: x[1])
    
    vehicle_weight = df_feat['LargeVehicles'].map({'Allowed': 1.0, 'Not Allowed': 0.5}).fillna(0.75)
    df_feat['Traffic_Capacity_Index'] = df_feat['NumberofLanes'] * vehicle_weight
    df_feat['interaction_geo_time'] = df_feat['geohash'].astype(str) + "_" + df_feat['time_slot_15m'].astype(str)
    
    return df_feat

X_train_fe = extract_features(X_train_lag)
X_test_fe = extract_features(X_test_lag)

 [ENGINEERING] Synthesizing cyclical time and spatial features...


In [5]:
print(" [ENGINEERING] DUAL Clustering (Generating both 25-Hub and 150-Hub representations)...")

global_coords = pd.concat([X_train_fe[['lat', 'lon']], X_test_fe[['lat', 'lon']]]).dropna()

kmeans_25 = KMeans(n_clusters=25, random_state=42, n_init=10).fit(global_coords)
X_train_fe['cluster_25'] = kmeans_25.predict(X_train_fe[['lat', 'lon']].fillna(0))
X_test_fe['cluster_25'] = kmeans_25.predict(X_test_fe[['lat', 'lon']].fillna(0))

kmeans_150 = KMeans(n_clusters=150, random_state=42, n_init=10).fit(global_coords)
X_train_fe['cluster_150'] = kmeans_150.predict(X_train_fe[['lat', 'lon']].fillna(0))
X_test_fe['cluster_150'] = kmeans_150.predict(X_test_fe[['lat', 'lon']].fillna(0))

center_lat, center_lon = global_coords['lat'].mean(), global_coords['lon'].mean()
X_train_fe['distance_to_center'] = np.sqrt((X_train_fe['lat'] - center_lat)**2 + (X_train_fe['lon'] - center_lon)**2)
X_test_fe['distance_to_center'] = np.sqrt((X_test_fe['lat'] - center_lat)**2 + (X_test_fe['lon'] - center_lon)**2)

 [ENGINEERING] DUAL Clustering (Generating both 25-Hub and 150-Hub representations)...


In [6]:
print(" [ENGINEERING] Computing Target Encodings (Using RAW Target)...")

# Notice we use y_raw here to keep feature distributions physically accurate
def get_oof_encoding(train_df, test_df, target_array, col, folds=5):
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)
    train_enc = np.zeros(len(train_df))
    w_train, w_test = train_df.copy(), test_df.copy()
    w_train["_tgt_"] = target_array
    global_mean = target_array.mean()
    
    test_enc = w_test[col].map(w_train.groupby(col)["_tgt_"].mean()).fillna(global_mean).values
    for tr_idx, val_idx in kf.split(w_train):
        fold_map = w_train.iloc[tr_idx].groupby(col)["_tgt_"].mean()
        train_enc[val_idx] = w_train.iloc[val_idx][col].map(fold_map).fillna(global_mean).values
    return train_enc, test_enc

X_train_fe['gh_demand_mean'], X_test_fe['gh_demand_mean'] = get_oof_encoding(X_train_fe, X_test_fe, y_raw, 'geohash')
X_train_fe['TE_geo_time'], X_test_fe['TE_geo_time'] = get_oof_encoding(X_train_fe, X_test_fe, y_raw, 'interaction_geo_time')

cat_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
for c in cat_cols:
    le = LabelEncoder()
    le.fit(X_train_fe[c].astype(str).tolist() + X_test_fe[c].astype(str).tolist())
    X_train_fe[c] = le.transform(X_train_fe[c].astype(str))
    X_test_fe[c] = le.transform(X_test_fe[c].astype(str))

features = [c for c in X_train_fe.columns if c not in ['timestamp', 'interaction_geo_time', 'Index']]
X, X_test = X_train_fe[features].values, X_test_fe[features].values

 [ENGINEERING] Computing Target Encodings (Using RAW Target)...


In [7]:
print(" [TOWER 1] Training Generalization Models (Log-Space Target)...")

lgb_p1 = {'objective': 'regression', 'metric': 'rmse', 'learning_rate': 0.03, 'max_depth': 8, 'min_child_samples': 20, 'verbose': -1, 'random_state': 42, 'n_jobs': -1}
xgb_p1 = {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'learning_rate': 0.03, 'max_depth': 7, 'random_state': 42, 'n_jobs': -1}
cat_p1 = {'iterations': 2000, 'learning_rate': 0.03, 'depth': 8, 'eval_metric': 'RMSE', 'verbose': 0, 'random_seed': 42}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_lgb_t1, test_lgb_t1 = np.zeros(len(X)), np.zeros(len(X_test))
oof_xgb_t1, test_xgb_t1 = np.zeros(len(X)), np.zeros(len(X_test))
oof_cat_t1, test_cat_t1 = np.zeros(len(X)), np.zeros(len(X_test))

for fold, (t_idx, v_idx) in enumerate(kf.split(X)):
    # V6 Core: Models train on y_log
    X_tr, y_tr_log, X_va, y_va_log = X[t_idx], y_log[t_idx], X[v_idx], y_log[v_idx]
    
    m_lgb = lgb.train(lgb_p1, lgb.Dataset(X_tr, y_tr_log), 2500, valid_sets=[lgb.Dataset(X_va, y_va_log)], callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_lgb_t1[v_idx] = np.expm1(m_lgb.predict(X_va)) # Instantly revert back to raw space
    test_lgb_t1 += np.expm1(m_lgb.predict(X_test)) / 5
    
    m_xgb = xgb.train(xgb_p1, xgb.DMatrix(X_tr, y_tr_log), 2500, evals=[(xgb.DMatrix(X_va, y_va_log), 'val')], early_stopping_rounds=100, verbose_eval=False)
    oof_xgb_t1[v_idx] = np.expm1(m_xgb.predict(xgb.DMatrix(X_va)))
    test_xgb_t1 += np.expm1(m_xgb.predict(xgb.DMatrix(X_test))) / 5
    
    m_cat = CatBoostRegressor(**cat_p1).fit(X_tr, y_tr_log, eval_set=(X_va, y_va_log))
    oof_cat_t1[v_idx] = np.expm1(m_cat.predict(X_va))
    test_cat_t1 += np.expm1(m_cat.predict(X_test)) / 5

# Optimize weights using raw predictions against raw targets
def obj_t1(w):
    w = np.array(w)
    if w.sum() == 0: return 999.0
    w_norm = w / w.sum()
    blend = (w_norm[0] * oof_lgb_t1) + (w_norm[1] * oof_xgb_t1) + (w_norm[2] * oof_cat_t1)
    return -max(0, 100 * r2_score(y_raw, blend))

w_t1 = minimize(obj_t1, [0.33, 0.33, 0.33], method='Nelder-Mead').x
w_t1 /= sum(w_t1)
preds_tower_1 = np.clip((w_t1[0]*test_lgb_t1 + w_t1[1]*test_xgb_t1 + w_t1[2]*test_cat_t1), 0.0, 1.0)
print(f" Tower 1 (Log-Space Generalization) Locked. True R2 Score: {max(0, 100 * r2_score(y_raw, (w_t1[0]*oof_lgb_t1 + w_t1[1]*oof_xgb_t1 + w_t1[2]*oof_cat_t1))):.4f}")

 [TOWER 1] Training Generalization Models (Log-Space Target)...
 Tower 1 (Log-Space Generalization) Locked. True R2 Score: 95.7263


In [8]:
print(" [TOWER 2] Training Deep Memory Models (Log-Space Target)...")

lgb_p2 = {'objective': 'regression', 'metric': 'rmse', 'learning_rate': 0.03, 'max_depth': 12, 'min_child_samples': 10, 'verbose': -1, 'random_state': 42, 'n_jobs': -1}
xgb_p2 = {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'learning_rate': 0.03, 'max_depth': 9, 'random_state': 42, 'n_jobs': -1}
cat_p2 = {'iterations': 2500, 'learning_rate': 0.03, 'depth': 10, 'eval_metric': 'RMSE', 'verbose': 0, 'random_seed': 42}

oof_lgb_t2, test_lgb_t2 = np.zeros(len(X)), np.zeros(len(X_test))
oof_xgb_t2, test_xgb_t2 = np.zeros(len(X)), np.zeros(len(X_test))
oof_cat_t2, test_cat_t2 = np.zeros(len(X)), np.zeros(len(X_test))

for fold, (t_idx, v_idx) in enumerate(kf.split(X)):
    X_tr, y_tr_log, X_va, y_va_log = X[t_idx], y_log[t_idx], X[v_idx], y_log[v_idx]
    
    m_lgb = lgb.train(lgb_p2, lgb.Dataset(X_tr, y_tr_log), 3000, valid_sets=[lgb.Dataset(X_va, y_va_log)], callbacks=[lgb.early_stopping(150, verbose=False)])
    oof_lgb_t2[v_idx] = np.expm1(m_lgb.predict(X_va))
    test_lgb_t2 += np.expm1(m_lgb.predict(X_test)) / 5
    
    m_xgb = xgb.train(xgb_p2, xgb.DMatrix(X_tr, y_tr_log), 3000, evals=[(xgb.DMatrix(X_va, y_va_log), 'val')], early_stopping_rounds=150, verbose_eval=False)
    oof_xgb_t2[v_idx] = np.expm1(m_xgb.predict(xgb.DMatrix(X_va)))
    test_xgb_t2 += np.expm1(m_xgb.predict(xgb.DMatrix(X_test))) / 5
    
    m_cat = CatBoostRegressor(**cat_p2).fit(X_tr, y_tr_log, eval_set=(X_va, y_va_log))
    oof_cat_t2[v_idx] = np.expm1(m_cat.predict(X_va))
    test_cat_t2 += np.expm1(m_cat.predict(X_test)) / 5

def obj_t2(w):
    w = np.array(w)
    if w.sum() == 0: return 999.0
    w_norm = w / w.sum()
    blend = (w_norm[0] * oof_lgb_t2) + (w_norm[1] * oof_xgb_t2) + (w_norm[2] * oof_cat_t2)
    return -max(0, 100 * r2_score(y_raw, blend))

w_t2 = minimize(obj_t2, [0.33, 0.33, 0.33], method='Nelder-Mead').x
w_t2 /= sum(w_t2)
preds_tower_2 = np.clip((w_t2[0]*test_lgb_t2 + w_t2[1]*test_xgb_t2 + w_t2[2]*test_cat_t2), 0.0, 1.0)
print(f" Tower 2 (Log-Space Deep Memory) Locked. True R2 Score: {max(0, 100 * r2_score(y_raw, (w_t2[0]*oof_lgb_t2 + w_t2[1]*oof_xgb_t2 + w_t2[2]*oof_cat_t2))):.4f}")

 [TOWER 2] Training Deep Memory Models (Log-Space Target)...
 Tower 2 (Log-Space Deep Memory) Locked. True R2 Score: 95.8929


In [12]:
print(" [FUSION] Applying 70/30 Meta-Blend & Formatting Final Matrix...")

# V6 Core: 70% Generalization / 30% Deep Memory Weighting Strategy
ultimate_blend_preds = (preds_tower_1 * 0.70) + (preds_tower_2 * 0.30)

submission_df = pd.DataFrame({
    'Index': test_submission_index,
    'demand': ultimate_blend_preds
})

submission_df.to_csv("submission_v6.csv", index=False)

print("\n" + "+=+="*20)
print(" V6 LOG-TRANSFORM ENSEMBLE COMPLETE!")
print(f" FINAL SUBMISSION ROWS    : {submission_df.shape[0]} (Target: 41778)")
print(f" FINAL SUBMISSION COLUMNS : {list(submission_df.columns)}")
print(" Saved flawlessly as 'submission_v6.csv'. Ready for upload!")
print("+=+="*20)

 [FUSION] Applying 70/30 Meta-Blend & Formatting Final Matrix...

+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
 V6 LOG-TRANSFORM ENSEMBLE COMPLETE!
 FINAL SUBMISSION ROWS    : 41778 (Target: 41778)
 FINAL SUBMISSION COLUMNS : ['Index', 'demand']
 Saved flawlessly as 'submission_v6.csv'. Ready for upload!
+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
